# 🎬 Runpod Anime Subber — Jupyter Launcher
شغّل الخلايا بالترتيب داخل **JupyterLab** على Runpod:
- **الخلية 1:** سحب المشروع من GitHub + تثبيت الأدوات والفونتات.
- **الخلية 2:** ربط Google Drive في `rclone`.
- **الخلية 3:** تشغيل واجهة Gradio: ترجمة إنجليزي ← عربي عبر DeepSeek (ولو الحلقة خام، Whisper بيفرّغ الصوت تلقائياً)، حرق، ضغط HEVC بأعلى جودة وأقل حجم MP4، ورفع DoodStream / Google Drive. المفاتيح بتتقري تلقائياً من `subber_secrets.json` الموجود في المشروع.
- **الخلية 4:** تشغيل **متصفح Chrome داخل Runpod (noVNC على بورت 6080)** للرفع اليدوي من `/workspace/outputs`.


In [ ]:
# === الخلية 1: سحب المشروع من GitHub وتثبيت الأدوات والفونتات ===
import os

GITHUB_REPO = "https://github.com/ismailhesham/runpod-anime-subber.git"
REPO_DIR = "/workspace/runpod-anime-subber"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only || true
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

%cd {REPO_DIR}
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg rclone curl fontconfig > /dev/null
!mkdir -p ~/.local/share/fonts && cp -f fonts/*.ttf fonts/*.otf ~/.local/share/fonts/ && fc-cache -fv > /dev/null
!pip install -q -r requirements.txt
print("✅ تم التجهيز بنجاح!")


In [ ]:
# === الخلية 2: ربط Google Drive في rclone ===
import os

GDRIVE_TOKEN = '''{"access_token":"ya29.a0AehfTOltdbxD1w6cAzPoxmYdcWCvZ434mFWjzoBA3d1pe2timEdorYj7XFsDw7TK0jT6MPBuHa9rrwHxxm5a1gjSGXy0qvpzvnpuK1vFNCuTxMsDaqKGeBX6w8aspOBcIgK6AdjuWifsV8lGqLM5IDr1i8ZfqPL4TkKUkgicyjngnKtHs2J7xesl2-xUPo9y30ZKKMoaCgYKAVQSARYSFQHGX2MiBp4bUOzVuVur8rGjYofNBQ0206","token_type":"Bearer","refresh_token":"1//03_d6qYeLU12YCgYIARAAGAMSSQF-L9Ir-fBONRektIbB0YpfiFQqaDPas7iC--bXRQPEWlTJdKQ_-4N9RM43ftC64CGD0ilVKIkgRGAP2WnYM_Zcf5QUGr3hYh0","expiry":"2026-11-21T02:46:39.5670534+02:00","expires_in":3599}'''

if GDRIVE_TOKEN.strip():
    os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
    with open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w", encoding="utf-8") as f:
        f.write(f"[gdrive]\ntype = drive\nscope = drive\ntoken = {GDRIVE_TOKEN.strip()}\n")
    print("✅ تم حفظ rclone.conf (gdrive).")
else:
    print("⚠️ مفيش توكن Google Drive — الرفع على Drive مش هيشتغل.")


In [ ]:
# === الخلية 3: تشغيل واجهة Gradio (بورت 7860) ===
import os

os.environ["UI_USER"] = "admin"
os.environ["UI_PASS"] = "123456"
os.environ["GRADIO_SHARE"] = "true"
os.environ["PORT"] = "7860"
os.environ["RUNPOD_OUTPUT_ROOT"] = "/workspace/outputs"

!python3 /workspace/runpod-anime-subber/app.py


In [ ]:
# === الخلية 4: تشغيل متصفح Chrome داخل Runpod (noVNC على بورت 6080) في الخلفية ===
import os, json

# تحميل المفاتيح المحفوظة على البود (من الخلية 3)
if os.path.isfile("/workspace/subber_secrets.json"):
    for k, v in json.load(open("/workspace/subber_secrets.json")).items():
        if v:
            os.environ[k] = v

os.environ.setdefault("DOODSTREAM_API_KEY", "582507jbhtz0ispf1ub44e")
os.environ["UI_USER"] = "admin"
os.environ["UI_PASS"] = "123456"
os.environ["GRADIO_SHARE"] = "true"
os.environ["PORT"] = "7860"
os.environ["RUNPOD_OUTPUT_ROOT"] = "/workspace/outputs"

!python3 /workspace/runpod-anime-subber/app.py
